# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eman123-123/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

I score the **held-out test split** (the same client-grouped rows from w05/w06 — the model never
trained on these) with the winning w05 model (logistic regression) and turn its probability into
one of four actions, each with a plain-language reason code:

- **refresh_priority_1** (`declining_page1_review`) — model probability ≥ 0.6 AND the page already
  ranks on page 1 (avg position ≤ 10). Highest priority: it's visible enough that a refresh has
  somewhere to pay off.
- **refresh_review** (`declining_review`) — model probability ≥ 0.6 but not yet ranking well.
  Worth a human look, lower urgency than the page-1 group.
- **schedule_refresh** (`stale_but_visible`) — lower model probability, but content hasn't been
  updated in 270+ days and still gets above-median impressions — the Week-4 rule baseline's own
  signal, kept as a second net for pages the model scores as currently "fine."
- **monitor** / **no_action** — everything else, split by whether probability is merely moderate
  or clearly low-risk (<0.3).


In [1]:
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ml-internship"):
        os.system("git clone -q https://github.com/Eman123-123/flyrank-ml-internship.git")
    os.chdir("flyrank-ml-internship")

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["target_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)

raw_numeric = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
]
categorical_features = ["competition_level", "content_type", "main_intent"]


def add_features(frame):
    f = frame.copy()
    f["has_position"] = (f["avg_position"] > 0).astype(int)
    f["avg_position_clean"] = f["avg_position"].where(f["avg_position"] > 0, np.nan)
    for c in raw_numeric[5:]:
        f[f"log_{c}"] = np.log1p(f[c].clip(lower=0))
    f["has_keyword_data"] = f["search_volume"].notna().astype(int)
    f["has_word_count"] = f["word_count"].notna().astype(int)
    return f


log_numeric = [f"log_{c}" for c in raw_numeric[5:]]
final_numeric = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position_clean", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "has_position", "has_keyword_data", "has_word_count",
] + log_numeric

# Same client-grouped split as w05/w06 -- the queue below is built on TEST rows only,
# i.e. pages the model never saw during training.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
tr_idx, te_idx = next(gss.split(df, groups=df["client_id"]))
train_df, test_df = df.iloc[tr_idx].copy(), df.iloc[te_idx].copy()
train_p, test_p = add_features(train_df), add_features(test_df)

X_train, y_train = train_p[final_numeric + categorical_features], train_p["target_declining"]
X_test = test_p[final_numeric + categorical_features]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), final_numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
pipe = Pipeline([("prep", preprocess),
                  ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED))])
pipe.fit(X_train, y_train)
test_p["model_proba"] = pipe.predict_proba(X_test)[:, 1]

# ---- build the ranked action queue ----
d = test_p.copy()
has_pos = d["avg_position"] > 0
d["reason_code"] = "monitor"
d.loc[(d["model_proba"] >= 0.6) & has_pos & (d["avg_position"] <= 10), "reason_code"] = "declining_page1_review"
d.loc[(d["model_proba"] >= 0.6) & ~(has_pos & (d["avg_position"] <= 10)), "reason_code"] = "declining_review"
d.loc[
    (d["model_proba"] < 0.6) & (d["days_since_last_update"] >= 270)
    & (d["impressions_90d"] >= d["impressions_90d"].median()),
    "reason_code",
] = "stale_but_visible"
d.loc[d["model_proba"] < 0.3, "reason_code"] = "healthy_low_risk"

action_map = {
    "declining_page1_review": "refresh_priority_1",
    "declining_review": "refresh_review",
    "stale_but_visible": "schedule_refresh",
    "monitor": "monitor",
    "healthy_low_risk": "no_action",
}
d["action"] = d["reason_code"].map(action_map)
d["score"] = (d["model_proba"] * 100).round(1)

action_queue = d.sort_values("score", ascending=False)[
    ["content_id", "client_id", "score", "action", "reason_code",
     "avg_position", "impressions_90d", "days_since_last_update", "ctr"]
].reset_index(drop=True)

print("Action counts:")
print(action_queue["action"].value_counts())
print("\nTop 10:")
action_queue.head(10)


Action counts:
action
monitor               3538
refresh_priority_1    1793
refresh_review        1103
no_action              681
Name: count, dtype: int64

Top 10:


,content_id,client_id,score,action,reason_code,avg_position,impressions_90d,days_since_last_update,ctr
0,content_82107ddb4e14,client_8b940be7fb,95.1,refresh_priority_1,declining_page1_review,6.2,1599,20,0.50
1,content_142a53c529dc,client_8b940be7fb,93.5,refresh_priority_1,declining_page1_review,3.2,6703,20,0.30
2,content_e5f459e737b7,client_f369cb89fc,93.0,refresh_priority_1,declining_page1_review,5.9,56363,20,0.01
3,content_d6e1bbb4a996,client_d029fa3a95,92.4,refresh_priority_1,declining_page1_review,3.9,4955,20,0.00
4,content_e60afc334f4b,client_d029fa3a95,92.2,refresh_review,declining_review,13.1,2744,20,0.11
5,content_c82bc0c24241,client_f369cb89fc,92.2,refresh_priority_1,declining_page1_review,4.3,13676,8,0.00
6,content_8ba781dafa55,client_8527a891e2,92.0,refresh_priority_1,declining_page1_review,9.0,16156,104,0.00
7,content_5d5653c4eb4f,client_4e07408562,91.7,refresh_priority_1,declining_page1_review,5.7,15101,7,0.00
8,content_823ea9b9b355,client_f369cb89fc,91.4,refresh_priority_1,declining_page1_review,3.9,4369,20,0.00
9,content_acaab4530f70,client_d029fa3a95,91.2,refresh_priority_1,declining_page1_review,3.7,6570,20,0.02


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses it:** a content or SEO reviewer with a limited weekly review budget, using the queue to
decide which existing pages to open first — not an automated publishing or de-indexing system.

**Where it's valid:** on the starter dataset's snapshot window and client mix, as a *ranking* to
triage attention, backed by a model that beat the Week-4 baseline on precision@50 (0.74 vs 0.48,
w05/w06) on a client-held-out split.

**Where it stops being valid:** for any client/brand outside this 32-client starter slice
(untested — w06 shows the honest grouped-split number is noticeably lower than a naive random
split, so I don't assume this generalizes for free); for causal claims ("refreshing this page will
fix it" — this is a decline *risk* ranking, not a refresh-outcome predictor); beyond the 90-day
snapshot window (no time-based validation was run, only client-grouped); and for any page whose
`reason_code` depends on `avg_position == 0` — those are "no data," not "great data," and I flag
them via `has_position` rather than silently zero-filling in scoring.


In [2]:
# Validity boundary, stated as a check rather than just prose:
print(f"Test clients (the only population this queue is validated on): {test_df['client_id'].nunique()}")
print(f"Rows with avg_position == 0 (no data, excluded from position-based reasoning): "
      f"{(test_df['avg_position'] == 0).sum()} / {len(test_df)}")
print(f"Base rate on this test split: {test_df['target_declining'].mean():.3f} "
      f"-- any action rate should be read against this, not against 50%.")


Test clients (the only population this queue is validated on): 8
Rows with avg_position == 0 (no data, excluded from position-based reasoning): 64 / 7115
Base rate on this test split: 0.517 -- any action rate should be read against this, not against 50%.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**A human must check, before acting on any `refresh_priority_1` / `refresh_review` row:**
whether the apparent decline is a seasonal or one-off traffic dip rather than a structural
problem; whether `avg_position` is real (`avg_position > 0`) before trusting a "page 1" claim;
and whether the page's `content_type` / `main_intent` make a refresh worthwhile at all (e.g. a
`feedly article` with no keyword data is a different kind of asset than a `keyword article`).

**Never automated from this output:**
- Auto-publishing or auto-editing content from the model's score alone.
- Auto-deprioritizing / de-indexing a page because it scored `healthy_low_risk` — that label
  means "lower modeled risk in this snapshot," not "verified fine forever."
- Any cross-client comparison that implies one client's content team is under- or
  over-performing another's — `client_id` is a pseudonym for grouping only, never a scoreboard.


In [3]:
# No-go checklist as a small assertion, so "Run All" enforces it rather than just asserting in prose.
forbidden_as_features = ["trend_pct", "trend_direction", "content_id", "client_id"]
used_as_features = final_numeric + categorical_features
leak_hit = [c for c in forbidden_as_features if c in used_as_features]
print("Forbidden columns present in scoring features:", leak_hit if leak_hit else "none - clean")
print(f"\n'healthy_low_risk' rows: {(action_queue['action'] == 'no_action').sum()} "
      f"-- reminder: this means lower modeled risk THIS snapshot, not a permanent clearance.")


Forbidden columns present in scoring features: none - clean

'healthy_low_risk' rows: 681 -- reminder: this means lower modeled risk THIS snapshot, not a permanent clearance.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Base-rate drift:** if the declining-proxy rate on new data moves far from the 0.517 seen here,
  the model's probability thresholds (0.6 / 0.3) need re-tuning, not blind reuse.
- **Precision@50 drop on a fresh holdout:** re-run the w05/w06 comparison on new data periodically;
  if precision@50 falls back toward the 0.48 baseline or the 0.517 base rate, the model has stopped
  adding value and should be retrained or retired.
- **New clients:** the model has only been validated on the 32 clients in the starter dataset,
  8 of them held out. A new client with a very different traffic profile should be treated as
  unvalidated until it appears in a fresh evaluation.
- **Feature drift:** if `avg_position == 0` (no-data) rates, or missing-keyword-data rates, shift
  a lot from this snapshot's levels, the imputation choices baked into `preprocess` may no longer
  be appropriate.


In [4]:
# Snapshot the trigger thresholds so a future run can diff against them.
monitoring_snapshot = {
    "base_rate_declining_test": round(float(test_df["target_declining"].mean()), 3),
    "precision_at_50_model": 0.74,   # from w05/w06, same split
    "precision_at_50_baseline": 0.48,  # from w05, same split
    "n_clients_validated": int(test_df["client_id"].nunique()) + int(train_df["client_id"].nunique()),
    "avg_position_no_data_rate": round(float((df["avg_position"] == 0).mean()), 4),
}
print(monitoring_snapshot)


{'base_rate_declining_test': 0.517, 'precision_at_50_model': 0.74, 'precision_at_50_baseline': 0.48, 'n_clients_validated': 32, 'avg_position_no_data_rate': 0.0402}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Per `work/README.md`: CSVs under `work/` are gitignored (no datasets in git), so the queue itself
stays local/regenerable, but the **summary numbers** go into a small committed JSON — that's the
receipt the paper's numbers trace back to.


In [5]:
import json
import os

os.makedirs("work/outputs", exist_ok=True)

# Full queue -- local only, regenerable, gitignored (work/**/*.csv)
action_queue.to_csv("work/outputs/action_queue.csv", index=False)

# Small committed summary -- the receipts (work/README.md: metrics JSONs stay committed)
summary = {
    "model": "logistic_regression",
    "split": "client_grouped_holdout",
    "n_test_rows": int(len(action_queue)),
    "n_test_clients": int(test_df["client_id"].nunique()),
    "base_rate_test": round(float(test_df["target_declining"].mean()), 3),
    "precision_at_50_model": 0.74,
    "precision_at_50_baseline_week4": 0.48,
    "action_counts": action_queue["action"].value_counts().to_dict(),
    **monitoring_snapshot,
}
with open("work/outputs/w07_action_playbook_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Wrote work/outputs/action_queue.csv (", len(action_queue), "rows )")
print("Wrote work/outputs/w07_action_playbook_summary.json")
print(json.dumps(summary, indent=2))


Wrote work/outputs/action_queue.csv ( 7115 rows )
Wrote work/outputs/w07_action_playbook_summary.json
{
  "model": "logistic_regression",
  "split": "client_grouped_holdout",
  "n_test_rows": 7115,
  "n_test_clients": 8,
  "base_rate_test": 0.517,
  "precision_at_50_model": 0.74,
  "precision_at_50_baseline_week4": 0.48,
  "action_counts": {
    "monitor": 3538,
    "refresh_priority_1": 1793,
    "refresh_review": 1103,
    "no_action": 681
  },
  "base_rate_declining_test": 0.517,
  "precision_at_50_baseline": 0.48,
  "n_clients_validated": 32,
  "avg_position_no_data_rate": 0.0402
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.